<a href="https://colab.research.google.com/github/DhimanTarafdar/AAAA/blob/main/audio_mantra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install gtts pydub -q
!apt-get install -y ffmpeg -q
print("✅ Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
✅ Done!


In [3]:
from gtts import gTTS
from pydub import AudioSegment
from pydub.effects import normalize
from IPython.display import Audio, display
from google.colab import files
import os


def make_spiritual_audio(input_mp3_path):
    """
    Target: Warm, devotional Hindu Female Pujari voice
    — Standard, professional, spiritually immersive —
    (ভক্তিমূলক, হৃদয়স্পর্শী, মন্দিরের পরিবেশের মতো অনুভূতি)

    KEY CHANGES vs original:
    ─────────────────────────────────────────────────────────
    1. Pitch  : +2.8 semi (was +4.2) → more natural female, not cartoon-high
    2. Speed  : 0.93 (was 0.88)     → devotional but still intelligible
    3. EQ     : shelving approach instead of hard low-pass
                  • low_pass  3400 Hz  (was 2800) → less muffled, voice clarity
                  • high_pass  160 Hz  (was 120)  → tighter, less rumble
    4. Reverb : progressive pre-delay + cascaded decay
                Simulates a stone-walled temple room, not a bathroom echo
    5. Warmth boost: +2 dB around 500–800 Hz region via gentle shelf
    6. Silence: tighter head/tail for cleaner playback
    7. Fade    : slightly longer fade-in for a meditative arrival feel
    ─────────────────────────────────────────────────────────
    """
    audio = AudioSegment.from_mp3(input_mp3_path)

    # ── 1. Pitch Shift → Natural soft female Pujari tone ─────────────────
    # +2.8 semitones ≈ 237–245 Hz range — warm, clear, authoritative female
    pitch_semitones = 2.8
    new_rate = int(audio.frame_rate * (2 ** (pitch_semitones / 12.0)))
    audio = (
        audio._spawn(audio.raw_data, overrides={"frame_rate": new_rate})
        .set_frame_rate(audio.frame_rate)
    )

    # ── 2. Tempo → Devotional pace, still intelligible ───────────────────
    # 0.93 = ~93% speed → calm & deliberate, not dragging
    tempo_factor = 0.93
    audio = (
        audio._spawn(
            audio.raw_data,
            overrides={"frame_rate": int(audio.frame_rate * tempo_factor)}
        )
        .set_frame_rate(audio.frame_rate)
    )

    # ── 3. EQ Shaping → Clear, warm, non-muffled ─────────────────────────
    # Remove low rumble (breath pops, background noise)
    audio = audio.high_pass_filter(160)

    # Keep speech clarity — 3400 Hz is the sweet spot:
    # cuts harshness & sibilance without muddying the voice
    audio = audio.low_pass_filter(3400)

    # Subtle warmth boost: bring up 500–800 Hz "body" of the voice
    # Pydub doesn't have parametric EQ, so we layer a slightly boosted
    # band-limited version of the audio (+1.8 dB, gentle shelf effect)
    warm_band = audio.low_pass_filter(900).high_pass_filter(400)
    audio = audio.overlay(warm_band - 14)   # blend at -14 dBFS for subtlety

    # ── 4. Temple Reverb → Stone hall, not bathroom echo ─────────────────
    #
    # Strategy: pre-delay + 5 cascaded reflections
    #   • Pre-delay (25 ms) = distance between speaker and first wall
    #   • Each reflection arrives later and decays more
    #   • Decay curve follows natural room physics (not linear)
    #
    # Reflection table: (pre_delay_ms, total_delay_ms, decay_dB)
    reflections = [
        (25,  55,  11),   # 1st reflection  — closest wall
        (25,  110, 17),   # 2nd reflection  — side wall
        (25,  200, 23),   # 3rd reflection  — back wall
        (25,  340, 29),   # 4th reflection  — ceiling bounce
        (25,  520, 36),   # 5th reflection  — far diffuse tail
    ]

    result = audio
    for pre_ms, total_ms, decay_db in reflections:
        # Silence padding simulates the pre-delay before each reflection arrives
        echo = AudioSegment.silent(duration=total_ms) + (audio - decay_db)

        # Pad the shorter segment so overlay lengths match
        diff = len(echo) - len(result)
        if diff > 0:
            result = result + AudioSegment.silent(duration=diff)
        else:
            echo = echo + AudioSegment.silent(duration=-diff)

        result = result.overlay(echo, position=0)

    audio = result

    # ── 5. Gentle final air cut → remove reverb harshness ────────────────
    audio = audio.low_pass_filter(4000)

    # ── 6. Subtle volume trim → sits at listening level, not loud ─────────
    audio = audio - 2

    # ── 7. Silence padding → meditative breathing room ───────────────────
    audio = (
        AudioSegment.silent(duration=1000)   # 1s lead-in
        + audio
        + AudioSegment.silent(duration=2500) # 2.5s tail (reverb decay)
    )

    # ── 8. Fade in/out → smooth, devotional arrival & departure ──────────
    audio = audio.fade_in(1100).fade_out(2000)

    # ── 9. Final normalization → consistent loudness ──────────────────────
    audio = normalize(audio)

    return audio


def generate_mantra(mantra_text, filename):
    """
    Generate a spiritually processed audio file from a Devanagari mantra.

    Args:
        mantra_text (str): The mantra in Devanagari script
        filename    (str): Output filename (without .mp3 extension)

    Returns:
        str: Path to the saved .mp3 file
    """
    print(f"\n🕉️  Processing: '{filename}'")

    # ── TTS Generation ────────────────────────────────────────────────────
    # slow=True gives gTTS a more measured, deliberate pace —
    # combined with our tempo shift this creates a natural devotional rhythm
    tts = gTTS(text=mantra_text, lang='hi', slow=True)
    raw_path = f"_raw_{filename}.mp3"
    tts.save(raw_path)
    print(f"   🎙️  TTS generated")

    # ── Apply devotional processing ───────────────────────────────────────
    audio = make_spiritual_audio(raw_path)

    # ── Export at high quality ────────────────────────────────────────────
    out_path = f"{filename}.mp3"
    audio.export(out_path, format="mp3", bitrate="320k")

    duration = len(audio) / 1000
    print(f"   ✅ Saved : {out_path}  ({duration:.1f}s)")

    # ── Preview in Colab ──────────────────────────────────────────────────
    display(Audio(out_path))

    # ── Download ──────────────────────────────────────────────────────────
    files.download(out_path)
    print(f"   ⬇️  Downloading...")

    # ── Cleanup temp file ─────────────────────────────────────────────────
    if os.path.exists(raw_path):
        os.remove(raw_path)

    return out_path


# ─────────────────────────────────────────────────────────────────────────────
# Quick test — uncomment and run to verify
# ─────────────────────────────────────────────────────────────────────────────
# generate_mantra("ॐ नमः शिवाय", "om_namah_shivaya")
# generate_mantra("ॐ गं गणपतये नमः", "ganesh_mantra")

print("✅ Devotional Female Pujari Voice — Fine-Tuned & Ready!")
print()
print("📌 Changes from original:")
print("   • Pitch  : +2.8 semi  (was +4.2) → natural, not cartoon-high")
print("   • Tempo  : 0.93x      (was 0.88) → devotional but intelligible")
print("   • LPF    : 3400 Hz    (was 2800) → clear voice, not muffled")
print("   • HPF    : 160 Hz     (was 120)  → less rumble")
print("   • Reverb : 5-reflection temple model (was simple echo stack)")
print("   • gTTS   : slow=True             (was False) → measured pace")
print("   • Warmth : gentle 400–900 Hz body boost")

✅ Devotional Female Pujari Voice — Fine-Tuned & Ready!

📌 Changes from original:
   • Pitch  : +2.8 semi  (was +4.2) → natural, not cartoon-high
   • Tempo  : 0.93x      (was 0.88) → devotional but intelligible
   • LPF    : 3400 Hz    (was 2800) → clear voice, not muffled
   • HPF    : 160 Hz     (was 120)  → less rumble
   • Reverb : 5-reflection temple model (was simple echo stack)
   • gTTS   : slow=True             (was False) → measured pace
   • Warmth : gentle 400–900 Hz body boost


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [4]:
# ================================================================
# 🕉️  এখানে তোমার মন্ত্র গুলো লেখো
#
#  "file_name"  : "মন্ত্র text"
#
#  ✅ যত খুশি মন্ত্র add করো
#  ✅ প্রতিটা আলাদা .mp3 file এ save হবে
#  ✅ নাম তুমি নিজে দেবে
# ================================================================

MANTRAS = {

    "surya_mantra":
        "ॐ ,जवाकुसुम ,सङ्काशं ,काश्यपेयं ,महद्युतिम् । तमोरिं ,सर्वपापघ्नं ,प्रणतोऽस्मि ,दिवाकरम् ॥",

    "gayatri_mantra":
        "ॐ ,भूर्भुवः ,स्वः । तत्सवितुर्वरेण्यं ,भर्गो ,देवस्य ,धीमहि । धियो, यो ,नः ,प्रचोदयात् ॥",

    "mahamrityunjaya":
        "ॐ ,त्र्यम्बकं ,यजामहे ,सुगन्धिं ,पुष्टिवर्धनम् । उर्वारुकमिव ,बन्धनान् ,मृत्योर्मुक्षीय ,मामृतात् ॥",

    "shanti_path":
        "ॐ ,क्लीं ,कालिकायै ,नमः । पूर्वे ,रक्षतु, मामिन्द्राणी, आग्नेय्यां ,रक्षतु ,महेश्वरी, दक्षिणे, रक्षतु ,वाराही, नैर्ऋत्यां ,रक्षतु ,नारसिंही, पश्चिमे ,रक्षतु ,चामुण्डा, वायव्यां ,रक्षतु ,माहेश्वरी, उत्तरे ,रक्षतु ,वैष्णवी, ऐशान्यां ,रक्षतु ब्राह्मी, ऊर्ध्वं ,रक्षतु ,शिवदूती, अधो, रक्षतु त्रिपुरा ।",

    # ── নতুন মন্ত্র এভাবে add করো ──
    # "আমার_মন্ত্র": "মন্ত্রের text এখানে",

}

# ================================================================
# ▶️  সব মন্ত্র generate করো
# ================================================================

print(f"📋 মোট মন্ত্র: {len(MANTRAS)}টা")
print("=" * 50)

done = []
for filename, mantra_text in MANTRAS.items():
    try:
        f = generate_mantra(mantra_text, filename)
        done.append(f)
    except Exception as e:
        print(f"   ❌ Error in '{filename}': {e}")

print("\n" + "=" * 50)
print(f"🎉 Complete! {len(done)}/{len(MANTRAS)} files ready:")
for f in done:
    print(f"   ✅ {f}")

📋 মোট মন্ত্র: 4টা

🕉️  Processing: 'surya_mantra'
   🎙️  TTS generated
   ✅ Saved : surya_mantra.mp3  (14.4s)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   ⬇️  Downloading...

🕉️  Processing: 'gayatri_mantra'
   🎙️  TTS generated
   ✅ Saved : gayatri_mantra.mp3  (15.6s)


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

   ⬇️  Downloading...

🕉️  Processing: 'shanti_path'
   🎙️  TTS generated
   ✅ Saved : shanti_path.mp3  (39.9s)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   ⬇️  Downloading...

🎉 Complete! 4/4 files ready:
   ✅ surya_mantra.mp3
   ✅ gayatri_mantra.mp3
   ✅ mahamrityunjaya.mp3
   ✅ shanti_path.mp3
